# 101 - A First Diffraction Model (Pupil to PSF)


For most problems, diffraction can be well modeled as a simple propagation from
a pupil to the point spread function.  In Goodman's variable notation, this
process begins by building the complex pupil function $P$:

$$ P(\xi, \eta) = A(\xi,\eta) \cdot \exp\left(i \tfrac{2\pi}{\lambda}\, \phi(\xi,\eta) \right) $$

$A$ is the amplitude; a circle or some other shape.  $\phi$ is the phase, in
units of length.  in prysm, this is nanometers.  The PSF is then calculated
as the square of the absolute value of the fourier transform.

We'll build $A$ and $\phi$, assemble $P$, and propagate it to a PSF, for a
10 mm f/10 lens at HeNe wavelength.

In [ ]:
import numpy as np
from matplotlib import pyplot as plt

from prysm.coordinates import make_xy_grid, cart_to_polar
from prysm.geometry import circle
from prysm.propagation import Wavefront
from prysm.wavelengths import HeNe
from prysm.polynomials import hopkins

# a 10 mm diameter, f/10 lens (100 mm efl) at the HeNe wavelength
EPD = 10.0          # entrance pupil diameter, mm
EFL = 100.0         # focal length, mm
FNO = EFL / EPD     # f/10
WVL = HeNe          # 0.6328 um

xi, eta = make_xy_grid(256, diameter=EPD)
r, t = cart_to_polar(xi, eta)
dx = xi[0, 1] - xi[0, 0]            # pupil sample spacing, mm
A = circle(EPD / 2, r)             # the clear circular aperture
airy_radius = 1.22 * WVL * FNO     # first dark ring, microns

See the foundational college if you need help with `make_xy_grid` or `cart_to_polar`.

`airy_radius` is going to be used later to compare and set plot limits,
it is not needed for the propagation model.

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(A, extent=[-EPD/2, EPD/2, -EPD/2, EPD/2], cmap='gray')
ax.set(xlabel='xi [mm]', ylabel='eta [mm]', title='amplitude A');

With $A$ in hand, we just need $\phi$.  We'll use the Hopkins polynomials to
add $W_{040}$, or $\rho^4$; spherical aberration:

In [ ]:
rho = r / (EPD / 2)
# W040 spherical, unit coefficient; trailing 0 is field height
# (which does not actually matter for W040)
phi = hopkins(0, 4, 0, rho, t, 0)

phi_display = phi.copy()
phi_display[A != 1] = np.nan        # blank outside the aperture (for display only)
fig, ax = plt.subplots(figsize=(4, 4))
im = ax.imshow(phi_display, extent=[-EPD/2, EPD/2, -EPD/2, EPD/2], cmap='RdBu')
fig.colorbar(im, ax=ax, fraction=0.046)
ax.set(xlabel='xi [mm]', ylabel='eta [mm]', title='phase ϕ');

Now we assemble the pupil function with `Wavefront.from_amp_and_phase`, which
combines $A$ and $\phi$ into the complex $P$.  We'll first use `phase=None`
to make a diffraction-limited PSF.  When propagating, your options are to
use `.focus` or `.focus_dft`.  The former uses a fast fourier transform, which
is the conventional propagation technique for most software but is losing
relevance for many modern problems.  When using `.focus` you need to use the
`Q` parameter, which sets the ratio of the number of samples in the point
spread function to the pupil function.  For an intensity PSF, you need at
minimum `Q>=2` to avoid aliasing.  We won't use `focus_dft` in this tutorial,
and will use `Q=4` for a visually smoother result.  If you want to compute
something like an MTF, only `Q=2` is needed; there will be no aliasing and
the extra padding will only produce more zeros beyond the optical cutoff in the
MTF plane.

In [ ]:
phi_500 = phi * 500.0   # nm, zero-to-peak

wf_perfect = Wavefront.from_amp_and_phase(A, None, WVL, dx)   # phase None -> ideal
E = wf_perfect.focus(EFL, Q=4)   # pupil -> PSF plane, Q=4 for fine sampling
psf = E.intensity                # |E|^2, the incoherent PSF

psf.plot2d(xlim=airy_radius * 8, power=1/3, cmap='gray', axis_labels=('focal x, um', 'focal y, um'))
plt.title('diffraction-limited PSF (Airy pattern)')

`power=1/3` on the plot is used as a stretch to make the peripheral rings visible.

We then repeat with aberration:

In [ ]:
wf_aberrated = Wavefront.from_amp_and_phase(A, phi_500, WVL, dx)
psf_ab = wf_aberrated.focus(EFL, Q=4).intensity

psf_ab.plot2d(xlim=airy_radius * 8, power=1/3, cmap='gray', axis_labels=('focal x, um', 'focal y, um'))
plt.title('PSF with 500 nm of spherical aberration')

## Wrap-up

This notebook covered the simple pupil to focus propagation that covers the
majority of applications.  The remaining notebooks in the 100 series cover the
building blocks for models.  If you want to model a broadband system or care
about radiometry, see notebook 201.  The later 300 series covers coronagraph
propagation, and the 400 series differentiable models.

Hopkins polynomials were used for example, but you will probably want Zernike
polynomials most of the time.  Those are covered in the polynomials college's
101 notebook.